# Big Data Lab 5 — Semi-Structured Data: Nested JSON + Schema Drift

Most real-world “big data” is not a perfect table. It arrives as **semi-structured JSON**: nested objects, arrays, missing keys, and fields that change over time (**schema drift**).

In this lab you will take a real semi-structured source (**GH Archive**) and build a small, practical pipeline that produces **analysis-ready Parquet tables**.

## What you’ll build (end-to-end)
- Download a few fixed **GH Archive** hours (`json.gz`) into an immutable **bronze** folder.
- Build a stable **silver events header** table (typed columns + a contract).
- Normalize nested JSON into two tables:
  - **push commits** (one row per commit from `PushEvent`)
  - **pull requests** (one row per PR event from `PullRequestEvent`)
- Detect at least one **schema drift** symptom and implement a mitigation.
- Publish a small **gold** table (per-repo per-day metrics) you can query quickly.

## Why this matters
If you can reliably turn semi-structured JSON into stable tables, you can:
- handle messy inputs without your pipeline breaking,
- explain tradeoffs (schema inference vs contracts, normalize vs keep JSON),
- and build datasets that downstream analytics can trust.

---

## Evaluation
**50% correctness + 50% explanation**

Be ready to explain:
- what “semi-structured” means,
- what schema drift is (give one example you observed),
- why we keep bronze,
- why we store an “escape hatch” JSON column,
- why we normalize arrays/objects into tables.

## Concepts cheat sheet (read once)
- **NDJSON / JSONL**: newline-delimited JSON (one JSON object per line).
- **Semi-structured**: not a fixed table; fields can be optional and nested.
- **Schema inference**: the reader guesses types/columns from data samples.
- **Schema contract**: what *your* table guarantees (required columns, types, nullability).
- **Schema drift**: fields appear/disappear, or types change across time/partitions.
- **Normalize**: turn nested arrays/objects into relational tables.
- **Escape hatch**: keep raw JSON (or a JSON string) so you can re-parse later.

## Setup (run once)

In [4]:
!pip -q install duckdb pyarrow matplotlib

import json
import shutil
import subprocess
import time
import uuid
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

print('duckdb', duckdb.__version__)
print('pyarrow', pa.__version__)

duckdb 1.3.2
pyarrow 18.1.0


## Optional: Persist outputs in Google Drive
If you want your outputs to survive a runtime reset, mount Drive and set `BASE_DIR` there.

In [5]:
from google.colab import drive  # type: ignore

USE_DRIVE = False  # TODO: set True if you want persistence

if USE_DRIVE:
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/BDLab/Lab5_GHArchive_JSON')
else:
    BASE_DIR = Path('/content/BDLab/Lab5_GHArchive_JSON')

BRONZE_DIR = BASE_DIR / 'bronze_raw'
SILVER_EVENTS_DIR = BASE_DIR / 'silver_events'
SILVER_COMMITS_DIR = BASE_DIR / 'silver_push_commits'
SILVER_PRS_DIR = BASE_DIR / 'silver_pull_requests'
GOLD_DIR = BASE_DIR / 'gold_repo_day_metrics'
AUX_DIR = BASE_DIR / '_aux'

for d in [BRONZE_DIR, SILVER_EVENTS_DIR, SILVER_COMMITS_DIR, SILVER_PRS_DIR, GOLD_DIR, AUX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BASE_DIR

PosixPath('/content/BDLab/Lab5_GHArchive_JSON')

In [6]:
import glob


def has_parquet_files(glob_pattern: str) -> bool:
    return len(glob.glob(glob_pattern)) > 0


def ensure_parquet_files(glob_pattern: str, dataset_name: str):
    if not has_parquet_files(glob_pattern):
        raise RuntimeError(
            f"No files found for {dataset_name}: {glob_pattern}\n"
            "Run section 10 ('Run the pipeline') first in this runtime, then retry."
        )


## 1) Ingest bronze: download fixed GH Archive hours
We’ll process a couple of **recent hours** plus one **older hour** to make schema drift discussions concrete.

Format:
`https://data.gharchive.org/YYYY-MM-DD-H.json.gz` (hour is **not** zero-padded).

In [7]:
# Recent hours (same date as Lab 4 by default)
NEW_DATE_PREFIX = '2024-01-15'
NEW_HOURS = [12, 13]

# One older hour (used for drift comparison)
OLD_DATE_PREFIX = '2018-01-15'
OLD_HOURS = [12]

HOUR_KEYS = [
    *[f"{NEW_DATE_PREFIX}-{h}" for h in NEW_HOURS],
    *[f"{OLD_DATE_PREFIX}-{h}" for h in OLD_HOURS],
]


def gharchive_url(hour_key: str) -> str:
    return f"https://data.gharchive.org/{hour_key}.json.gz"


def bronze_path(hour_key: str) -> Path:
    return BRONZE_DIR / f"{hour_key}.json.gz"


def download_if_missing(hour_key: str) -> Path:
    out = bronze_path(hour_key)
    if out.exists() and out.stat().st_size > 0:
        print(f"Already present: {out.name} ({out.stat().st_size/1024/1024:.1f} MB)")
        return out

    url = gharchive_url(hour_key)
    print('Downloading', url)
    subprocess.run(['wget', '-q', '-O', str(out), url], check=True)

    if not out.exists() or out.stat().st_size == 0:
        raise RuntimeError(f"Download failed for {hour_key}")

    print(f"Saved: {out.name} ({out.stat().st_size/1024/1024:.1f} MB)")
    return out


for hk in HOUR_KEYS:
    download_if_missing(hk)

pd.DataFrame({
    'source_hour': HOUR_KEYS,
    'file_mb': [bronze_path(hk).stat().st_size/1024/1024 for hk in HOUR_KEYS],
}).sort_values('source_hour')

Already present: 2024-01-15-12.json.gz (117.6 MB)
Already present: 2024-01-15-13.json.gz (125.0 MB)
Already present: 2018-01-15-12.json.gz (23.2 MB)


,source_hour,file_mb
2,2018-01-15-12,23.210420
0,2024-01-15-12,117.566851
1,2024-01-15-13,124.999067


### Tiny exercise 1 (5–10 minutes)
Add **one more hour** to `NEW_HOURS` (for example `14`).

Answer in 2–4 sentences:
- What do you expect to change (runtime, number of rows, event-type variety)?
- Do you expect schema drift to be easier or harder to observe?

## 2) Inspect raw data safely (semi-structured shape)
Goal: make nested JSON tangible without loading everything into Python.

We’ll use DuckDB to scan compressed NDJSON directly.

In [8]:
con = duckdb.connect(database=':memory:')

hk = HOUR_KEYS[0]
raw = bronze_path(hk).as_posix()

# Event type distribution (top 15)
con.execute(f"""
SELECT type, COUNT(*) AS n
FROM read_json_auto('{raw}', format='newline_delimited', ignore_errors=true)
GROUP BY 1
ORDER BY n DESC
LIMIT 15
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,type,n
0,PushEvent,200257
1,CreateEvent,26381
2,PullRequestEvent,15886
3,IssueCommentEvent,10144
4,WatchEvent,8669
5,DeleteEvent,6747
6,PullRequestReviewEvent,5996
7,IssuesEvent,3666
8,PullRequestReviewCommentEvent,3482
9,ForkEvent,2083


In [9]:
# Show nested type shapes (STRUCT/LIST/...) in the inferred schema
hk = HOUR_KEYS[0]
raw = bronze_path(hk).as_posix()

con.execute(f"""
SELECT
  typeof(repo) AS repo_t,
  typeof(actor) AS actor_t,
  typeof(payload) AS payload_t
FROM read_json_auto('{raw}', format='newline_delimited', ignore_errors=true)
LIMIT 1
""").df()

,repo_t,actor_t,payload_t
0,"STRUCT(id BIGINT, ""name"" VARCHAR, url VARCHAR)","STRUCT(id BIGINT, login VARCHAR, display_login...","STRUCT(""action"" VARCHAR, issue STRUCT(url VARC..."


In [10]:
# Peek at payload as JSON text (escape-hatch view)
hk = HOUR_KEYS[0]
raw = bronze_path(hk).as_posix()

con.execute(f"""
SELECT
  type,
  substr(to_json(payload), 1, 240) || '...' AS payload_json_preview
FROM read_json_auto('{raw}', format='newline_delimited', ignore_errors=true)
LIMIT 5
""").df()

,type,payload_json_preview
0,IssueCommentEvent,"{""action"":""created"",""issue"":{""url"":""https://ap..."
1,DeleteEvent,"{""action"":null,""issue"":null,""comment"":null,""re..."
2,PushEvent,"{""action"":null,""issue"":null,""comment"":null,""re..."
3,CreateEvent,"{""action"":null,""issue"":null,""comment"":null,""re..."
4,IssueCommentEvent,"{""action"":""created"",""issue"":{""url"":""https://ap..."


### Tiny exercise 2 (5–10 minutes)
Pick one nested field that might drift over time.

Example ideas:
- `payload.pull_request.draft`
- `payload.pull_request.merged`
- `payload.issue.state_reason`

Task:
- Compute how often the field is missing (`NULL`) for the relevant event type, **per hour**.
- Write 2–4 sentences explaining why missing fields are normal in semi-structured data.

In [11]:
# TODO (exercise): compute missing rate of your chosen nested field per hour.
# Hint: use json_extract(to_json(payload), '$.your.path.here') and filter to the relevant event type.

## 3) Output layout + a safe Parquet write helper
We write one Parquet file per partition, then rename it into place.

This is notebook-friendly: it avoids leaving a half-written partition if something crashes mid-write.

In [12]:
def write_partition_atomic(base_dir: Path, partition_key: str, partition_value: str, table: pa.Table):
    final_dir = base_dir / f"{partition_key}={partition_value}"
    tmp_dir = base_dir / f".tmp_{partition_key}={partition_value}_{uuid.uuid4().hex}"
    bak_dir = base_dir / f".bak_{partition_key}={partition_value}_{uuid.uuid4().hex}"

    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    pq.write_table(table, tmp_dir / 'part-00000.parquet', compression='snappy')
    (tmp_dir / '_SUCCESS').write_text('', encoding='utf-8')

    if final_dir.exists():
        if bak_dir.exists():
            shutil.rmtree(bak_dir)
        final_dir.rename(bak_dir)

    tmp_dir.rename(final_dir)

    if bak_dir.exists():
        shutil.rmtree(bak_dir)

    return final_dir

## 4) Process one hour at a time (read JSON once)
We avoid scanning the same gzip file repeatedly.

We create a DuckDB temp table `raw_hour` for one hour, then build our outputs from it.

Design note (important):
- We store an **escape hatch** column `payload_json` (a JSON string).
- This is more robust than storing a nested `STRUCT` in Parquet, because nested schemas can drift and become hard to union across partitions.

In [13]:
CORE_INVALID_PRED = "(event_id IS NULL OR event_id='' OR event_type IS NULL OR created_at_ts IS NULL OR repo_name IS NULL OR actor_login IS NULL OR actor_login='')"


def materialize_raw_hour(hour_key: str):
    raw = bronze_path(hour_key).as_posix()

    con.execute('DROP TABLE IF EXISTS raw_hour')
    con.execute(f"""
    CREATE TEMP TABLE raw_hour AS
    SELECT
      '{hour_key}' AS source_hour,
      CAST(id AS VARCHAR) AS event_id,
      CAST(type AS VARCHAR) AS event_type,
      TRY_CAST(created_at AS TIMESTAMP) AS created_at_ts,
      CAST(repo.name AS VARCHAR) AS repo_name,
      CAST(actor.login AS VARCHAR) AS actor_login,
      payload,
      to_json(payload) AS payload_json
    FROM read_json_auto('{raw}', format='newline_delimited', ignore_errors=true)
    """)

    rows_in = con.execute('SELECT COUNT(*) FROM raw_hour').fetchone()[0]
    print('raw_hour rows:', rows_in)

## 5) Silver table #1: stable events header (+ escape hatch)
Silver contract:
- required: `event_id`, `event_type`, `created_at_ts`, `repo_name`, `actor_login`

We keep `payload_json` in silver as an escape hatch.

In [14]:
def build_silver_events_for_hour(hour_key: str):
    table = con.execute(f"""
    SELECT
      source_hour,
      event_id,
      event_type,
      created_at_ts,
      repo_name,
      actor_login,
      payload_json
    FROM raw_hour
    WHERE NOT {CORE_INVALID_PRED}
    """).arrow()

    return write_partition_atomic(SILVER_EVENTS_DIR, 'source_hour', hour_key, table)

## 6) Normalize arrays: PushEvent → one row per commit
`PushEvent` contains a commits array. We turn that array into rows.

Implementation approach:
- Use DuckDB’s nested types directly: `UNNEST(payload.commits)`.
- Extract fields from the commit struct.

In [15]:
def build_push_commits_for_hour(hour_key: str):
    table = con.execute(f"""
    WITH push AS (
      SELECT *
      FROM raw_hour
      WHERE event_type='PushEvent' AND NOT {CORE_INVALID_PRED}
    )
    SELECT
      source_hour,
      event_id,
      created_at_ts,
      repo_name,
      actor_login,
      c.sha AS commit_sha,
      c.message AS commit_message,
      c.author.name AS author_name,
      c.author.email AS author_email
    FROM push
    CROSS JOIN UNNEST(payload.commits) AS t(c)
    """).arrow()

    return write_partition_atomic(SILVER_COMMITS_DIR, 'source_hour', hour_key, table)

### Tiny exercise 3 (10 minutes)
Use the produced commits table to compute something insightful.

Example:
- Average commit message length per repo (top 10 repos by commits).

Write 2–4 sentences explaining what you see (skew/outliers, why some repos dominate, etc.).

In [16]:
# TODO (exercise): write a DuckDB query over `silver_push_commits`.
# Hint: use read_parquet on the glob: SILVER_COMMITS_DIR / 'source_hour=*' / '*.parquet'

## 7) Normalize nested structs: PullRequestEvent → one row per PR event
`PullRequestEvent` has a nested `pull_request` object. We extract a stable set of columns.

Note:
- For some fields, direct struct access may be brittle across years.
- For drift-prone fields, you can always fall back to `payload_json` (escape hatch).

In [17]:
def build_pull_requests_for_hour(hour_key: str):
    table = con.execute(f"""
    WITH pr AS (
      SELECT *
      FROM raw_hour
      WHERE event_type='PullRequestEvent' AND NOT {CORE_INVALID_PRED}
    )
    SELECT
      source_hour,
      event_id,
      created_at_ts,
      repo_name,
      actor_login,
      CAST(payload.action AS VARCHAR) AS action,
      TRY_CAST(payload.pull_request.id AS BIGINT) AS pr_id,
      TRY_CAST(payload.pull_request.number AS BIGINT) AS pr_number,
      CAST(payload.pull_request.state AS VARCHAR) AS pr_state,
      TRY_CAST(payload.pull_request.merged AS BOOLEAN) AS merged,
      -- Drift-prone field: draft (may be missing in older data) => read from JSON escape hatch
      CASE lower(replace(CAST(json_extract(payload_json, '$.pull_request.draft') AS VARCHAR), '"', ''))
        WHEN 'true' THEN TRUE
        WHEN 'false' THEN FALSE
        ELSE NULL
      END AS draft,
      TRY_CAST(payload.pull_request.additions AS BIGINT) AS additions,
      TRY_CAST(payload.pull_request.deletions AS BIGINT) AS deletions,
      TRY_CAST(payload.pull_request.changed_files AS BIGINT) AS changed_files
    FROM pr
    """).arrow()

    return write_partition_atomic(SILVER_PRS_DIR, 'source_hour', hour_key, table)

## 8) Schema drift: detect it + implement a mitigation
You need two things:
1) A **drift proof**: show one field that behaves differently across the older vs newer hour.
2) A **mitigation**: explain (and show) why your pipeline doesn’t break.

Mitigation options (any one is acceptable):
- Use an escape hatch (`payload_json`) + null-tolerant parsing (`json_extract`, `TRY_CAST`)
- Keep a strict silver header contract and re-parse for specialized tables
- Restrict your model to stable fields and treat drift-prone fields as optional

In [18]:
# Schema drift report: PullRequestEvent field $.pull_request.draft across hours
# We compute a per-hour profile and persist it for Assignment B.

drift_rows = []

for hk in HOUR_KEYS:
    raw = bronze_path(hk).as_posix()

    q = f"""
    WITH pr AS (
      SELECT to_json(payload) AS payload_json
      FROM read_json_auto('{raw}', format='newline_delimited', ignore_errors=true)
      WHERE type = 'PullRequestEvent'
    )
    SELECT
      '{hk}' AS source_hour,
      COUNT(*) AS pr_events,
      SUM(CASE WHEN json_extract(payload_json, '$.pull_request.draft') IS NULL THEN 1 ELSE 0 END) AS draft_missing,
      SUM(CASE WHEN lower(replace(CAST(json_extract(payload_json, '$.pull_request.draft') AS VARCHAR), '"', '')) = 'true' THEN 1 ELSE 0 END) AS draft_true,
      SUM(CASE WHEN lower(replace(CAST(json_extract(payload_json, '$.pull_request.draft') AS VARCHAR), '"', '')) = 'false' THEN 1 ELSE 0 END) AS draft_false
    FROM pr
    """

    row = con.execute(q).df().iloc[0].to_dict()
    pr_events = int(row['pr_events'])
    missing = int(row['draft_missing'])
    row['draft_missing_rate'] = (missing / pr_events) if pr_events > 0 else None
    drift_rows.append(row)

schema_drift_report = pd.DataFrame(drift_rows).sort_values('source_hour').reset_index(drop=True)
out_csv = AUX_DIR / 'schema_drift_report.csv'
schema_drift_report.to_csv(out_csv, index=False)

print('Saved:', out_csv)
schema_drift_report

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: /content/BDLab/Lab5_GHArchive_JSON/_aux/schema_drift_report.csv


,source_hour,pr_events,draft_missing,draft_true,draft_false,draft_missing_rate
0,2018-01-15-12,3396,3396.0,0.0,0.0,1.0
1,2024-01-15-12,15886,0.0,299.0,15587.0,0.0
2,2024-01-15-13,17604,0.0,316.0,17288.0,0.0


## 9) Gold: per-repo per-day metrics
We publish a compact report table that’s fast to query.

Example metrics:
- number of commits (from PushEvent commits)
- number of PR events and merges
- merge rate

In [19]:
def build_gold_repo_day_metrics():
    commits_glob = (SILVER_COMMITS_DIR / 'source_hour=*' / '*.parquet').as_posix()
    prs_glob = (SILVER_PRS_DIR / 'source_hour=*' / '*.parquet').as_posix()

    ensure_parquet_files(commits_glob, 'silver_push_commits')
    ensure_parquet_files(prs_glob, 'silver_pull_requests')

    con.execute('DROP TABLE IF EXISTS gold')
    con.execute(f"""
    CREATE TEMP TABLE gold AS
    WITH commits AS (
      SELECT
        CAST(date_trunc('day', created_at_ts) AS DATE) AS day,
        repo_name,
        COUNT(*) AS push_commits
      FROM read_parquet('{commits_glob}')
      GROUP BY 1, 2
    ),
    prs AS (
      SELECT
        CAST(date_trunc('day', created_at_ts) AS DATE) AS day,
        repo_name,
        COUNT(*) AS pr_events,
        SUM(CASE WHEN merged THEN 1 ELSE 0 END) AS prs_merged
      FROM read_parquet('{prs_glob}')
      GROUP BY 1, 2
    )
    SELECT
      COALESCE(c.day, p.day) AS day,
      COALESCE(c.repo_name, p.repo_name) AS repo_name,
      COALESCE(push_commits, 0) AS push_commits,
      COALESCE(pr_events, 0) AS pr_events,
      COALESCE(prs_merged, 0) AS prs_merged,
      CASE WHEN COALESCE(pr_events, 0) = 0 THEN NULL ELSE prs_merged * 1.0 / pr_events END AS merge_rate
    FROM commits c
    FULL OUTER JOIN prs p
      ON c.day = p.day AND c.repo_name = p.repo_name
    """)

    gold_df = con.execute('SELECT * FROM gold').df()
    for day, sub in gold_df.groupby('day'):
        table = pa.Table.from_pandas(sub, preserve_index=False)
        write_partition_atomic(GOLD_DIR, 'day', str(day), table)

    return gold_df

## 10) Run the pipeline (end-to-end)
This is the “main run”. After this, you should have bronze + silver + gold artifacts on disk.

In [ ]:
# Main run
for hk in HOUR_KEYS:
    print('\n=== Processing', hk, '===')
    download_if_missing(hk)
    materialize_raw_hour(hk)
    build_silver_events_for_hour(hk)
    build_push_commits_for_hour(hk)
    build_pull_requests_for_hour(hk)

_ = build_gold_repo_day_metrics()

print('Done.')



=== Processing 2024-01-15-12 ===
Already present: 2024-01-15-12.json.gz (117.6 MB)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

raw_hour rows: 286864


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...


=== Processing 2018-01-15-12 ===
Already present: 2018-01-15-12.json.gz (23.2 MB)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

raw_hour rows: 63463


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done.


## 11) Sanity-check the produced datasets

In [20]:
silver_glob = (SILVER_EVENTS_DIR / 'source_hour=*' / '*.parquet').as_posix()
commits_glob = (SILVER_COMMITS_DIR / 'source_hour=*' / '*.parquet').as_posix()
prs_glob = (SILVER_PRS_DIR / 'source_hour=*' / '*.parquet').as_posix()
gold_glob = (GOLD_DIR / 'day=*' / '*.parquet').as_posix()

checks = {}
for name, glob_pattern in [
    ('silver_events_rows', silver_glob),
    ('push_commits_rows', commits_glob),
    ('pull_requests_rows', prs_glob),
    ('gold_rows', gold_glob),
]:
    if has_parquet_files(glob_pattern):
        checks[name] = con.execute(f"SELECT COUNT(*) FROM read_parquet('{glob_pattern}')").fetchone()[0]
    else:
        checks[name] = None

checks


{'silver_events_rows': 623281,
 'push_commits_rows': 549890,
 'pull_requests_rows': 36886,
 'gold_rows': 134398}

In [21]:
# Example query: top repos by PR events
con.execute(f"""
SELECT repo_name, COUNT(*) AS pr_events
FROM read_parquet('{prs_glob}')
GROUP BY 1
ORDER BY pr_events DESC
LIMIT 10
""").df()

,repo_name,pr_events
0,kovrus/opentelemetry-collector-contrib,70
1,microsoft/winget-pkgs,67
2,actions-canary/ForkPRCanary,62
3,Telepedia/mediawiki,58
4,google-test/signclav2-probe-repo,53
5,collibra/opentelemetry-collector-contrib,53
6,anurag-harness/kafka,53
7,my-git9/opentelemetry-collector-contrib,50
8,SonarSourceIT/pr-decoration,48
9,SimaTankSAAS/nifi-1.4.0,44


# Mini-assignment (end of lab)

## Assignment A — Add a third normalized table (choose 1)
Pick one event type and build a new silver table, partitioned by `source_hour`:
- `IssuesEvent` → `silver_issues`
- `IssueCommentEvent` → `silver_issue_comments`
- `ReleaseEvent` → `silver_releases`

Requirements:
- Write Parquet partitioned by `source_hour`.
- Include: `event_id`, `repo_name`, `created_at_ts`, `source_hour`.

## Assignment B — Drift proof + mitigation
- Show one drift symptom across your hours.
- Explain your mitigation (why your pipeline doesn’t break).

## Assignment C — Oral prompts
- When would you keep nested columns vs normalize?
- What is schema drift? Give one example you observed.
- Why keep bronze? What’s the point of an escape hatch column?

In [22]:
# Assignment A solution: add IssuesEvent normalization -> silver_issues partitioned by source_hour

SILVER_ISSUES_DIR = BASE_DIR / 'silver_issues'
SILVER_ISSUES_DIR.mkdir(parents=True, exist_ok=True)


def build_issues_for_hour(hour_key: str):
    table = con.execute(f"""
    WITH issues AS (
      SELECT *
      FROM raw_hour
      WHERE event_type='IssuesEvent' AND NOT {CORE_INVALID_PRED}
    )
    SELECT
      source_hour,
      event_id,
      repo_name,
      created_at_ts,
      CAST(payload.action AS VARCHAR) AS action,
      TRY_CAST(payload.issue.id AS BIGINT) AS issue_id,
      TRY_CAST(payload.issue.number AS BIGINT) AS issue_number,
      CAST(payload.issue.state AS VARCHAR) AS issue_state,
      CAST(payload.issue.title AS VARCHAR) AS issue_title,
      CAST(payload.issue.user.login AS VARCHAR) AS issue_author_login
    FROM issues
    """).arrow()

    return write_partition_atomic(SILVER_ISSUES_DIR, 'source_hour', hour_key, table)


for hk in HOUR_KEYS:
    materialize_raw_hour(hk)
    build_issues_for_hour(hk)

issues_glob = (SILVER_ISSUES_DIR / 'source_hour=*' / '*.parquet').as_posix()

con.execute(f"""
SELECT source_hour, COUNT(*) AS issue_events
FROM read_parquet('{issues_glob}')
GROUP BY 1
ORDER BY 1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

raw_hour rows: 286864


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

raw_hour rows: 272954


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

raw_hour rows: 63463


,source_hour,issue_events
0,2018-01-15-12,2305
1,2024-01-15-12,3666
2,2024-01-15-13,4405


## Assignment B — Drift proof + mitigation (written answer)

**Drift symptom shown:** in `schema_drift_report.csv`, the field `$.pull_request.draft` has a different missing-rate across hours (especially older vs newer data). This demonstrates schema drift because the same logical field is not consistently present over time.

**Mitigation used:**
- We keep a strict, stable silver contract (`event_id`, `event_type`, `created_at_ts`, `repo_name`, `actor_login`, `payload_json`).
- Drift-prone nested fields are parsed from `payload_json` with null-tolerant logic (`json_extract` + `TRY_CAST`-style behavior), so missing keys become `NULL` instead of crashing writes.
- We normalize specialized tables independently (for example `silver_pull_requests`, `silver_issues`), which isolates drift to optional columns while preserving core pipeline reliability.

## Assignment C — Oral prompts (prepared answers)

1. **When keep nested columns vs normalize?**
Keep nested columns when the nested object is rarely queried, highly variable, or mostly used for traceability. Normalize when repeated analytical queries need specific nested fields, joins, filtering, or aggregations.

2. **What is schema drift? Example observed.**
Schema drift means structure or meaning changes over time (field appears/disappears, type changes, or nested path changes). In this lab, `pull_request.draft` is present more consistently in newer data and missing more often in older hours.

3. **Why keep bronze and why an escape hatch column?**
Bronze preserves immutable raw input for reproducibility, reprocessing, and audits. The escape hatch (`payload_json`) prevents data loss when contracts evolve: if a field is not yet modeled or drifts unexpectedly, we can still recover and re-parse it later without re-downloading raw data.

## What you should have at the end
- `bronze_raw/YYYY-MM-DD-H.json.gz` raw files.
- `silver_events/source_hour=YYYY-MM-DD-H/part-00000.parquet`.
- `silver_push_commits/source_hour=YYYY-MM-DD-H/part-00000.parquet`.
- `silver_pull_requests/source_hour=YYYY-MM-DD-H/part-00000.parquet`.
- `gold_repo_day_metrics/day=YYYY-MM-DD/part-00000.parquet`.
- A drift symptom you can show + a mitigation you can explain.

## References (optional)
- GH Archive: https://www.gharchive.org/
- Data directory: https://data.gharchive.org/
- DuckDB JSON: https://duckdb.org/docs/data/json/overview.html